# pipelines.extractor.models

> Pydantic models for structured information extraction

In [ ]:
# | default_exp pipelines.extractor.models

In [ ]:
# | hide
from nbdev.showdoc import *

In [ ]:
# | export

"""Pre-defined Pydantic models for common structured extraction tasks.

These models can be used with the Extractor.extract_structured() method to extract
structured information from documents. They serve as examples and can be customized
for specific use cases.
"""

from typing import List, Optional, Union
try:
    from pydantic import BaseModel, Field
except ImportError:
    raise ImportError("Pydantic is required. Install with: pip install pydantic")


# ==================== System Parameters ====================

from pydantic import BaseModel, Field
from typing import List, Union

class SystemParameter(BaseModel):
    name: str = Field(
        description="The type of parameter being measured (e.g., 'Power', 'Weight', 'Speed'). Do NOT include the value or unit here."
    )
    value: Union[int, float] = Field(
        description="The numeric value only. Extract numbers like 1000, 3, 2.5. Required field."
    )
    unit: str = Field(
        description="The unit of measurement only (e.g., 'mph', 'grams'). Do NOT include the value here."
    )
    name_of_system: str = Field(
        default="UNKNOWN",
        description="name of system or larger component associated with the property being measured - enter UNKNOWN if not listed"
    )

class ParamCollection(BaseModel):
    params: List[SystemParameter] = Field(
        default=[],
        description="List of all key operational and performance parameters found in the text"
    )


# ==================== Key Performance Parameters (KPP) ====================

class KPPValue(BaseModel):
    """A Key Performance Parameter with threshold and objective values."""
    parameter_name: str = Field(
        description="Name of the key performance parameter"
    )
    threshold: Optional[str] = Field(
        default=None,
        description="Threshold (minimum required) value with unit (e.g., '10 minutes', '1000 cubic inches')"
    )
    objective: Optional[str] = Field(
        default=None,
        description="Objective (desired) value with unit"
    )
    notes: Optional[str] = Field(
        default=None,
        description="Additional notes or conditions"
    )


class KPPCollection(BaseModel):
    """Collection of Key Performance Parameters from acquisition documents."""
    kpps: List[KPPValue] = Field(
        default=[],
        description="List of all KPPs with their threshold (T) and objective (O) values"
    )



# ==================== Metadata Extraction ====================

class DocumentMetadata(BaseModel):
    """High-level metadata about a document."""
    title: Optional[str] = Field(default=None, description="Document title")
    document_type: Optional[str] = Field(default=None, description="Type of document (e.g., 'CDD', 'ICD', 'Technical Manual')")
    year: Optional[int] = Field(default=None, description="Year document was created")
    version: Optional[str] = Field(default=None, description="Document version")
    classification: Optional[str] = Field(default=None, description="Classification level")
    system_name: Optional[str] = Field(default=None, description="Name of system described in document")
    domain: Optional[str] = Field(default=None, description="Operational domain (AIR, LAND, SEA, SPACE)")
    summary: Optional[str] = Field(default=None, description="Brief summary of document content")

# ==================== Entity Extraction ====================

class Entity(BaseModel):
    """A named entity extracted from text."""
    name: str = Field(description="The entity name or value")
    type: str = Field(description="Entity type (e.g., 'PERSON', 'ORGANIZATION', 'LOCATION', 'DATE')")
    context: Optional[str] = Field(
        default=None,
        description="Surrounding context or relationship"
    )


class EntityCollection(BaseModel):
    """Collection of entities extracted from a document."""
    entities: List[Entity] = Field(
        default=[],
        description="List of all named entities found in the document"
    )


# ==================== Example Prompts ====================

# These are example prompts that work well with the models above

SYSTEM_PARAMETERS_PROMPT = """You are a helpful assistant that analyzes technical documents and extracts performance and operational parameters
associated with technical systems and platforms.

EXTRACTION RULES:
1. Extract ONLY parameters with explicit numeric values AND units clearly stated in the text
2. DO NOT extract: form field codes, rating scales, stand-alone numbers without context, or list positions
3. Each parameter must have: name, value (numeric), unit (measurement), and name_of_system

THRESHOLD/OBJECTIVE HANDLING:
- If a parameter value is marked with (T) or (Threshold), append " (Threshold)" to the parameter name
- If a parameter value is marked with (O) or (Objective), append " (Objective)" to the parameter name
- Examples:
  * "100W (T)" → name: "Power (Threshold)", value: 100, unit: "W"
  * "500W (O)" → name: "Power (Objective)", value: 500, unit: "W"
  * "3lbs (Threshold)" → name: "Weight (Threshold)", value: 3, unit: "lbs"

MULTIPLE VALUES:
- If there are multiple values/units for the same parameter, create separate entries
- Use the same base parameter name with appropriate (Threshold)/(Objective) suffixes

<text>
{text}
</text>
"""

KPP_EXTRACTION_PROMPT = """Extract Key Performance Parameters (KPPs) from this acquisition document.

Look for parameters marked with:
- (T) or [T] or (Threshold) = Threshold (minimum required) value
- (O) or [O] or (Objective) = Objective (desired) value
- (T=O) = Same value for both threshold and objective

For each KPP found, extract:
1. Parameter name (what is being measured)
2. Threshold value with unit
3. Objective value with unit
4. Any additional context or conditions

Examples:
- "Power generation capability must not occupy greater than 1000 cubic inches (Threshold) and 1600 cubic inches (Objective)."
  → parameter_name: "Power generation capability volume", threshold: "1000 cubic inches", objective: "1600 cubic inches"

Text to analyze:

{text}
"""

KPP_EXTRACTION_PROMPT = """Extract Key Performance Parameters (KPPs) from this acquisition document.

Look for parameters marked with:
- (T) or [T] or (Threshold) = Threshold (minimum required) value
- (O) or [O] or (Objective) = Objective (desired) value
- (T=O) = Same value for both threshold and objective

For each KPP found, extract:
1. Parameter name (what is being measured)
2. Threshold value with unit
3. Objective value with unit
4. Any additional context or conditions

Examples:
- "Power generation capability must not occupy greater than 1000 cubic inches (Threshold) and 1600 cubic inches (Objective)."
  → parameter_name: "Power generation capability volume", threshold: "1000 cubic inches", objective: "1600 cubic inches"

Text to analyze:

{text}
"""


DOCUMENT_METADATA_PROMPT = """Extract high-level metadata from this document.

Look for:
- Document title (usually on first page)
- Document type (CDD, ICD, Technical Manual, etc.)
- Year of publication
- Version or revision number
- Classification level
- System name being described
- Operational domain (AIR, LAND, SEA, SPACE, CYBER, etc.)

Also provide a 1-2 sentence summary of what this document covers.

Document text:

{text}
"""




ENTITY_EXTRACTION_PROMPT = """Extract all named entities from this document.

For each entity found, identify:
1. name: The entity name or value
2. type: Entity type (PERSON, ORGANIZATION, LOCATION, DATE, and no other relevant types)
3. context: Surrounding context or relationships (optional)

Examples:
- name: "Department of Defense", type: "ORGANIZATION", context: "Document author"
- name: "July 2013", type: "DATE", context: "Publication date"
- name: "Fort Benning", type: "LOCATION", context: "Training facility"

Document text:
{text}
"""

In [ ]:
# | notest

from onprem import LLM
from onprem.pipelines import Extractor


In [ ]:
# |  notest

llm = LLM("anthropic/claude-sonnet-4-5", max_tokens=32000, mute_stream=True)

In [ ]:
# | notest


# Initialize
extractor = Extractor(llm)

# Test document
content = """
 Aircraft Specifications:
 - Maximum Speed: 500 mph
 - Cruise Speed (avg): 350 knots
 - Operational Range: 2000 nm
 - Payload Capacity: 5000 lbs
 - Power Output: 1500 W
 - Total Weight: 25000 kg
"""

# Extract with whitelist filtering
whitelist = {'range', 'cruise speed', 'weight', 'power'}

result = extractor.extract_parameters(
     content=content,
     parameter_whitelist=whitelist,
)

# Display results
print(f"Extracted {len(result['params'])} matching parameters:\n")
for p in result['params']:
    print(f"  {p['name']}: {p['value']} {p['unit']}")


# cruise speed: 350 knots -> matches "cruise speed"
# range: 2000 nm          -> matches "operational range"
# power: 1500 W           -> matches "power output"
# weight: 25000 kg        -> matches "total weight"

  Whitelist filtered params: 6 → 4 items
Extracted 4 matching parameters:

  cruise speed: 350 knots
  range: 2000 nm
  power: 1500 W
  weight: 25000 kg


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()